# BRAID v2 — flatmap cross-attention bridge (CortexMAE brain tokens -> unpooled CLIP tokens)

One subject only, for now (`SUBJECT` below). Frozen inputs on both sides — CortexMAE brain tokens
and CLIP unpooled tokens are both precomputed and cached already (`braidv2_flatmap_unpooled_clip_colab.ipynb`)
— only the cross-attention bridge itself is trained here.

**Architecture**: 257 learnable query vectors (one per CLIP token position, including the CLS
position — a direct 1:1 correspondence, not a compress-then-expand scheme) cross-attend into the
frozen CortexMAE brain tokens via a small Transformer decoder stack (self-attention among queries +
cross-attention into brain tokens + FFN, per layer — this is exactly what `nn.TransformerDecoderLayer`
already implements). Output projected to CLIP's 1024-dim token space, trained with MSE against the
real (frozen, precomputed) CLIP tokens for that trial. This mirrors Brain-IT's semantic branch
(direct query-to-token correspondence, L2 loss) rather than a fewer-queries-plus-expansion design —
predicting values already shaped like what a downstream CLIP-conditioned decoder expects, rather
than an intermediate representation needing a further learned expansion.

**Why session-chunked training, not a global shuffle**: each cached brain-token session tensor is
`750 × 1456 × 768` float32 ≈ 3.35GB. Holding many sessions in memory at once isn't practical, so
training streams one session at a time (load, shuffle its 750 trials, train, free, next session)
rather than assuming the full dataset fits in RAM.

In [ ]:
import os, glob, re, json, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.io import loadmat
from tqdm.notebook import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- config ---
DRIVE_ROOT = "/content/drive/MyDrive/braid2"
INPUTS_DIR = f"{DRIVE_ROOT}/inputs"
FLATMAP_TOKENS_DIR = f"{INPUTS_DIR}/flatmap_brain_tokens"
CLIP_UNPOOLED_DIR   = f"{INPUTS_DIR}/clip_embeds_unpooled"
CLIP_TARGET_CACHE_DIR = f"{INPUTS_DIR}/clip_targets_cache"
os.makedirs(CLIP_TARGET_CACHE_DIR, exist_ok=True)

SUBJECT = "subj01"          # one subject only, for now
TRIALS_PER_SESSION = 750
SHARD_SIZE = 1000            # matches the unpooled-CLIP notebook's sharding

CLIP_DIM     = 1024          # confirmed from openai/clip-vit-large-patch14 last_hidden_state
NUM_QUERIES  = 257           # matches CLIP's token count exactly (1 CLS + 256 patch), 1:1 correspondence
D_MODEL      = 768           # == brain-token dim (ViT-B hidden size), avoids an input projection
N_LAYERS     = 4
N_HEADS      = 8
FFN_DIM      = 3072
DROPOUT      = 0.1

BATCH_SIZE = 128             # bumped from 32 -- small model + A100 headroom, fewer/larger minibatches per session
EPOCHS     = 20
LR         = 1e-4
GRAD_CLIP_NORM = 1.0         # standard transformer-training safety net, was previously absent

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

device: cuda


## `outputs/` run folder

Same convention used throughout this project — created once, right after config, everything about
this run (checkpoint, loss curves, eval metrics, config snapshot) saved under it.

In [ ]:
def make_run_dir(approach, base=f"{DRIVE_ROOT}/outputs"):
    run_dir = f"{base}/{approach}_{time.strftime('%Y%m%d_%H%M%S')}"
    os.makedirs(run_dir, exist_ok=True)
    return run_dir


def save_run_config(run_dir, **hparams):
    with open(f"{run_dir}/config.json", "w") as f:
        json.dump(hparams, f, indent=2, default=str)


run_dir = make_run_dir("flatmap_cross_attention_bridge")
print("run_dir:", run_dir)

run_dir: /content/drive/MyDrive/braid2/outputs/flatmap_cross_attention_bridge_20260831_030418


## NSD experiment design (trial -> imgBrick id mapping)

Same `nsd_expdesign.mat` / `masterordering` / `subjectim` convention used throughout this project —
needed to map a (subject, session, trial) to the imgBrick id whose CLIP tokens we want as the
training target.

In [ ]:
EXP = f"{INPUTS_DIR}/nsd_expdesign.mat"
if not os.path.exists(EXP):
    import urllib.request
    urllib.request.urlretrieve(
        "https://natural-scenes-dataset.s3.amazonaws.com/nsddata/experiments/nsd/nsd_expdesign.mat", EXP)
mat = loadmat(EXP)

masterordering = mat["masterordering"].reshape(-1).astype(np.int64) - 1   # [30000] trial-slot order
subjectim      = mat["subjectim"].astype(np.int64) - 1                     # [8, 10000] slot -> imgBrick idx

subject_idx      = int(SUBJECT[-2:]) - 1
imgbrick_ids_all = subjectim[subject_idx, masterordering]                  # [30000] imgBrick id per trial

## Compact per-subject CLIP-target cache

The 73 unpooled-CLIP shards are ~526MB each (~38GB total) — too much to hold in memory at once and
wasteful to reload repeatedly during training. This extracts, once, just the rows this subject's
*currently available* sessions actually reference, into a single dict cached to Drive. If more
flatmap sessions get computed later, delete the cache file to force a rebuild covering them too.

In [ ]:
session_files = sorted(glob.glob(f"{FLATMAP_TOKENS_DIR}/{SUBJECT}_flatmap_tokens_session*.pt"))
session_nums  = [int(re.search(r"session(\d+)", f).group(1)) for f in session_files]
print(f"{SUBJECT}: {len(session_files)} flatmap sessions available:", session_nums)
assert len(session_files) >= 2, f"need at least 2 sessions (1 train + 1 val), found {len(session_files)}"

needed_ids = set()
for sess in session_nums:
    start = (sess - 1) * TRIALS_PER_SESSION
    needed_ids.update(imgbrick_ids_all[start:start + TRIALS_PER_SESSION].tolist())
print(f"{len(needed_ids)} unique images referenced across available sessions")

CLIP_TARGET_CACHE = f"{CLIP_TARGET_CACHE_DIR}/{SUBJECT}_clip_targets.pt"
if os.path.exists(CLIP_TARGET_CACHE):
    clip_by_imgbrick = torch.load(CLIP_TARGET_CACHE)
    print(f"loaded cached targets: {len(clip_by_imgbrick)} images")
else:
    needed_by_shard = {}
    for i in needed_ids:
        needed_by_shard.setdefault(i // SHARD_SIZE, []).append(i)

    clip_by_imgbrick = {}
    for shard, ids in tqdm(sorted(needed_by_shard.items()), desc="extracting CLIP targets"):
        shard_tensor = torch.load(f"{CLIP_UNPOOLED_DIR}/unpooled_clip_shard{shard:03d}.pt")   # [1000,257,1024] fp16
        for i in ids:
            clip_by_imgbrick[i] = shard_tensor[i % SHARD_SIZE].clone()
        del shard_tensor

    torch.save(clip_by_imgbrick, CLIP_TARGET_CACHE)
    print(f"built + cached targets: {len(clip_by_imgbrick)} images -> {CLIP_TARGET_CACHE}")

subj01: 40 flatmap sessions available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]
10000 unique images referenced across available sessions
loaded cached targets: 10000 images


## Local fp16 cache for brain-token sessions

`torch.load` on a session file straight from the Drive mount is slow (~3.35GB float32 over Drive's
FUSE mount), and training re-reads the *same* session files every epoch — 20 epochs means paying that
slow read 20 times over for no reason. This copies each needed session to local Colab disk once,
downcast to fp16 (halves the bytes too), so all 20 epochs read from fast local storage instead.
Ephemeral — rebuilds automatically on a fresh runtime, nothing lost by not persisting it.

In [ ]:
LOCAL_CACHE_DIR = "/content/flatmap_tokens_fp16_cache"
os.makedirs(LOCAL_CACHE_DIR, exist_ok=True)

def local_cache_path(sess):
    return f"{LOCAL_CACHE_DIR}/{SUBJECT}_session{sess:02d}_fp16.pt"


def drive_session_path(sess):
    return f"{FLATMAP_TOKENS_DIR}/{SUBJECT}_flatmap_tokens_session{sess:02d}.pt"


for sess in tqdm(session_nums, desc="caching sessions to local disk (fp16)"):
    dest = local_cache_path(sess)
    if os.path.exists(dest):
        continue
    brain_tokens = torch.load(drive_session_path(sess))   # slow Drive read, float32 -- paid once
    torch.save(brain_tokens.half(), dest)                  # fast local disk, float16
    del brain_tokens

total_local_gb = sum(os.path.getsize(local_cache_path(s)) for s in session_nums) / 1e9
print(f"local cache ready: {total_local_gb:.1f}GB across {len(session_nums)} sessions")

caching sessions to local disk (fp16):   0%|          | 0/40 [00:00<?, ?it/s]

local cache ready: 67.1GB across 40 sessions


## Train / validation split

Simplest reasonable split for a first pass: hold out the single most-recent available session for
validation, train on the rest.

In [ ]:
val_session_nums   = session_nums[-1:]
train_session_nums = session_nums[:-1]
print("train sessions:", train_session_nums)
print("val session:", val_session_nums)


def load_session(sess):
    """-> (brain_tokens [n_trials, n_brain_tokens, D] float32, clip_targets [n_trials, 257, 1024] float32)"""
    brain_tokens = torch.load(local_cache_path(sess)).float()   # fast local read (fp16), upcast for training
    start = (sess - 1) * TRIALS_PER_SESSION
    ids = imgbrick_ids_all[start:start + brain_tokens.shape[0]]
    clip_targets = torch.stack([clip_by_imgbrick[i] for i in ids]).float()

    # NaN/Inf appearing on epoch 1 for both train and val points at bad source data, not training
    # instability (gradual blowups usually show rising loss first) -- check directly rather than guess.
    if not torch.isfinite(brain_tokens).all():
        bad = (~torch.isfinite(brain_tokens)).sum().item()
        raise ValueError(f"session {sess}: {bad} non-finite values in brain_tokens (of {brain_tokens.numel()})")
    if not torch.isfinite(clip_targets).all():
        bad = (~torch.isfinite(clip_targets)).sum().item()
        raise ValueError(f"session {sess}: {bad} non-finite values in clip_targets (of {clip_targets.numel()})")

    return brain_tokens, clip_targets

train sessions: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39]
val session: [40]


## Model

`nn.TransformerDecoderLayer` already implements exactly the block this needs per layer:
self-attention among the queries, cross-attention into the brain-token memory, then a feedforward
block, each with residual + layernorm — no need to hand-roll multi-head attention.

In [ ]:
class CrossAttentionBridge(nn.Module):
    def __init__(self, brain_dim, clip_dim, num_queries, d_model, n_layers, n_heads, ffn_dim, dropout):
        super().__init__()
        self.query_embed = nn.Parameter(torch.randn(num_queries, d_model) * 0.02)
        self.brain_proj  = nn.Linear(brain_dim, d_model) if brain_dim != d_model else nn.Identity()
        self.brain_norm  = nn.LayerNorm(d_model)   # brain_proj is Identity when brain_dim==d_model (our case),
                                                     # so without this nothing normalizes CortexMAE's raw,
                                                     # uncharacterized output scale before it's used as
                                                     # cross-attention keys/values -- a likely NaN contributor
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ffn_dim,
            dropout=dropout, batch_first=True,
        )
        self.decoder     = nn.TransformerDecoder(decoder_layer, num_layers=n_layers)
        self.output_proj = nn.Linear(d_model, clip_dim)

    def forward(self, brain_tokens):
        """brain_tokens: [B, N_brain, brain_dim] -> [B, num_queries, clip_dim]"""
        B = brain_tokens.shape[0]
        memory  = self.brain_norm(self.brain_proj(brain_tokens))
        queries = self.query_embed.unsqueeze(0).repeat(B, 1, 1)
        out     = self.decoder(tgt=queries, memory=memory)
        return self.output_proj(out)


brain_dim = load_session(train_session_nums[0])[0].shape[-1]
print("brain token dim:", brain_dim)

model = CrossAttentionBridge(
    brain_dim=brain_dim, clip_dim=CLIP_DIM, num_queries=NUM_QUERIES,
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS, ffn_dim=FFN_DIM, dropout=DROPOUT,
).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

num_params = sum(p.numel() for p in model.parameters())
print(f"bridge params: {num_params / 1e6:.1f}M")

brain token dim: 768
bridge params: 38.8M


## Training loop

Session-chunked: one session's tensor in memory at a time, minibatched and shuffled within it.
Per-session progress via tqdm; per-epoch train/val MSE + cosine tracked and plotted at the end.

In [ ]:
KNOWN_BAD_SESSIONS = {("subj01", 11)}
session_files = [f for f, s in zip(session_files, session_nums) if (SUBJECT, s) not in KNOWN_BAD_SESSIONS]
session_nums  = [s for s in session_nums if (SUBJECT, s) not in KNOWN_BAD_SESSIONS]
print(f"{SUBJECT}: {len(session_files)} flatmap sessions available:", session_nums)

subj01: 39 flatmap sessions available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]


In [ ]:
def run_epoch(session_nums_, epoch, train: bool):
    model.train(train)
    total_loss, total_cos, n_batches = 0.0, 0.0, 0

    order = list(session_nums_)
    if train:
        random.shuffle(order)

    desc = f"[bridge] epoch {epoch}/{EPOCHS} ({'train' if train else 'val'})"
    for sess in tqdm(order, desc=desc, leave=False):
        brain_tokens, clip_targets = load_session(sess)
        perm = torch.randperm(brain_tokens.shape[0]) if train else torch.arange(brain_tokens.shape[0])

        for b in range(0, len(perm), BATCH_SIZE):
            idx = perm[b:b + BATCH_SIZE]
            brain_batch = brain_tokens[idx].to(DEVICE)
            clip_batch  = clip_targets[idx].to(DEVICE)

            with torch.set_grad_enabled(train):
                pred = model(brain_batch)
                loss = F.mse_loss(pred, clip_batch)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)   # standard safety net, was absent
                optimizer.step()

            with torch.no_grad():
                cos = F.cosine_similarity(pred, clip_batch, dim=-1).mean()

            total_loss += loss.item()
            total_cos  += cos.item()
            n_batches  += 1

        del brain_tokens, clip_targets

    return total_loss / n_batches, total_cos / n_batches


def save_progress():
    """Called every epoch, not just at the end -- a crash mid-training (e.g. the NaN check firing)
    would otherwise lose the checkpoint and loss history for every epoch completed so far."""
    torch.save(model.state_dict(), f"{run_dir}/bridge.pt")
    with open(f"{run_dir}/loss_history.json", "w") as f:
        json.dump({
            "train_loss": train_loss_hist, "train_cos": train_cos_hist,
            "val_loss": val_loss_hist, "val_cos": val_cos_hist,
        }, f, indent=2)


train_loss_hist, train_cos_hist, val_loss_hist, val_cos_hist = [], [], [], []
for epoch in range(1, EPOCHS + 1):
    train_loss, train_cos = run_epoch(train_session_nums, epoch, train=True)
    val_loss, val_cos     = run_epoch(val_session_nums, epoch, train=False)
    train_loss_hist.append(train_loss); train_cos_hist.append(train_cos)
    val_loss_hist.append(val_loss);     val_cos_hist.append(val_cos)
    print(f"epoch {epoch:02d}  train mse={train_loss:.4f} cos={train_cos:.4f}  "
          f"val mse={val_loss:.4f} cos={val_cos:.4f}")
    save_progress()

print(f"saved: {run_dir}/bridge.pt, {run_dir}/loss_history.json")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=300)
axes[0].plot(train_loss_hist, label="train"); axes[0].plot(val_loss_hist, label="val")
axes[0].set_title("bridge MSE"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(train_cos_hist, label="train"); axes[1].plot(val_cos_hist, label="val")
axes[1].set_title("bridge cosine similarity"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{run_dir}/bridge_loss.png", dpi=300, bbox_inches="tight"); plt.show()

[bridge] epoch 1/20 (train):   0%|          | 0/39 [00:00<?, ?it/s]

ValueError: session 11: 838656000 non-finite values in brain_tokens (of 838656000)

## Save run config

In [ ]:
save_run_config(
    run_dir, subject=SUBJECT, train_sessions=train_session_nums, val_sessions=val_session_nums,
    num_queries=NUM_QUERIES, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    ffn_dim=FFN_DIM, dropout=DROPOUT, batch_size=BATCH_SIZE, epochs=EPOCHS, lr=LR,
)
print("saved:", f"{run_dir}/config.json")

## Evals — CLS-token two-way identification on the held-out session

No downstream image decoder is wired up yet (see open items below), so image-quality metrics
(PixCorr/SSIM/FID/etc., as used in `braidv2_fmri_clip_bridge_colab.ipynb`) aren't applicable here.
What *is* directly measurable from predicted vs. real CLIP tokens alone is retrieval-style
identification — the same `corrcoef`-based two-way identification formula used for the image-space
evals elsewhere in this project, applied directly to the CLS-position (index 0) predicted vs. real
CLIP embeddings for every trial in the held-out validation session: for each trial, is the
prediction more correlated with its own true match than with the true embedding of every other
trial in the session? Reported as a fraction in [0, 1] (chance ≈ 0.5).

In [ ]:
@torch.no_grad()
def two_way_identification(preds, reals):
    """preds/reals: [N, D] -- returns mean identification accuracy in [0, 1], chance ~= 0.5."""
    preds = preds.cpu().numpy() if torch.is_tensor(preds) else preds
    reals = reals.cpu().numpy() if torch.is_tensor(reals) else reals
    r = np.corrcoef(reals, preds)
    r = r[:len(reals), len(reals):]
    congruents = np.diag(r)
    success = r < congruents
    return float(np.mean(np.sum(success, 0)) / (len(reals) - 1))


model.eval()
val_preds, val_reals = [], []
for sess in val_session_nums:
    brain_tokens, clip_targets = load_session(sess)
    for b in range(0, brain_tokens.shape[0], BATCH_SIZE):
        brain_batch = brain_tokens[b:b + BATCH_SIZE].to(DEVICE)
        pred = model(brain_batch).cpu()
        val_preds.append(pred)
        val_reals.append(clip_targets[b:b + BATCH_SIZE])
val_preds = torch.cat(val_preds)   # [N, 257, 1024]
val_reals = torch.cat(val_reals)   # [N, 257, 1024]

cls_ident = two_way_identification(val_preds[:, 0], val_reals[:, 0])           # CLS token only
mean_pool_ident = two_way_identification(val_preds.mean(1), val_reals.mean(1))  # mean over all 257 tokens

eval_results = {"cls_two_way_identification": cls_ident, "mean_pool_two_way_identification": mean_pool_ident}
print(f"CLS-token two-way identification:       {cls_ident:.3f}")
print(f"mean-pooled-token two-way identification: {mean_pool_ident:.3f}")

with open(f"{run_dir}/eval_metrics.json", "w") as f:
    json.dump(eval_results, f, indent=2)
print(f"saved: {run_dir}/eval_metrics.json")

### Open items / known simplifications

1. **One subject, whatever sessions happen to be computed so far** — by design, per this
   conversation's compute constraints. Extending to more subjects/sessions just means adding more
   flatmap sessions upstream and rerunning the target-cache cell (delete the cache file first so it
   rebuilds with the new sessions included).
2. **Single held-out session for validation/eval** — a minimal split, not a rigorous eval protocol.
   The proper shared1000 held-out test set (repeat-averaged, matching MindEye-style papers) still
   needs the dedicated flatmap generation this project's earlier conversation already flagged as
   requiring most/all of a subject's sessions.
3. **No pooled-embedding auxiliary loss** — this trains purely against the unpooled 257-token target.
   A pooled/projected auxiliary loss is a reasonable later addition, not required for this to work.
4. **No decoder/reconstruction step yet** — the eval above measures identification directly in CLIP
   token space; turning predicted tokens into actual reconstructed images (and thus the full
   PixCorr/SSIM/FID-style eval suite used elsewhere in this project) needs the downstream
   CLIP-conditioned decoder discussed earlier (Versatile Diffusion / modified SDXL /
   Kandinsky-with-custom-conditioning), not yet wired up here.